# Pilot · Steps 3–5 (training features, linear probes, evaluation on val)

**Step 3:** for the 200 train scenes, features of up to 10k labelled vertices each from Utonia, Sonata, Concerto, an untrained PTv3, lifted DINOv2 and the raw inputs, written to the session's temp disk (~27 GB; nothing data-derived is kept). **Step 4:** a linear probe per configuration (A, B, C, C-eq, S, S+, K, K+, K+-eq, R, R+, X): learning rate chosen on 20 held-out train scenes, then 3 seeds on all 200. **Step 5:** every probe on all 50 val scenes with the official ScanNet++ metric (top-1/top-3 mIoU, per class, seen vs never-seen vertices, bootstrap over scenes), cross-checked against the toolbox's own eval_semantic. The two choices from pilot_02 are set in the first code cell. No `SCANNETPP_TOKEN` needed.

**Before running**
- [ ] **Accelerator:** GPU T4 x2
- [ ] **Internet:** on
- [ ] **Secrets** (Add-ons → Secrets): `GITHUB_TOKEN` attached
- [ ] **Environment:** pin to the same image as the wheelhouse (Settings → Environment preference); install_env.py refuses a mismatched image
- [ ] **Input:** private dataset `appearance-pt-wheels` (output of m1_01)
- [ ] **Inputs:** private datasets `scannetpp-core-val`, `scannetpp-core-train-a` and `scannetpp-core-train-b`

**Run:** Save Version → Save & Run All (Commit) (expected about 2.5–3 h; approved by the user on 2026-10-26).

**Paste back:** the three SUMMARY blocks (step 3, 4, 5; full logs: /kaggle/working/logs/pilot_03_*.log).

*Generated by `scripts/make_notebooks.py`; edit the scripts, not this notebook.*

In [ ]:
# ---- run configuration: the only cell to edit ----
REPO = "eylulpelinkilic/appearance-pt"  # your private GitHub repo (owner/name)
REF = "main"                  # branch, tag or commit SHA; the resolved commit is logged

In [ ]:
"""Kaggle bootstrap: fetch this (private) repo at a fixed ref, then run entry points with logs.

Embedded verbatim in every notebook by scripts/make_notebooks.py; edit here, then regenerate.
The GitHub token comes from Kaggle Secrets and reaches git only through environment variables
(never argv, files or output), so it cannot leak into logs or tracebacks; git's error messages
are shown (a notebook would otherwise hide them) with the token redacted.
"""

import base64
import os
import pathlib
import shutil
import subprocess
import sys
import time

_GIT_DEBUG_VARS = ("GIT_TRACE", "GIT_TRACE_CURL", "GIT_CURL_VERBOSE", "GIT_TRACE_PACKET")


def _git(args, cwd=None, env=None, capture=False, redact=()):
    # stderr is captured: a notebook cell never shows a subprocess's stderr, only the exit code
    r = subprocess.run(["git", *args], cwd=cwd, env=env, text=True, capture_output=True)
    if r.returncode != 0:
        err = (r.stderr or "").strip()
        for s in redact:
            err = err.replace(s, "***")
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode}):\n{err}")
    return r.stdout.strip() if capture else None


def clone_repo(repo, ref, dest="/tmp/appearance-pt", get_secret=None):
    """Fetch `repo` ("owner/name") at `ref` (branch, tag or commit) into `dest`.

    Submodules (public) are fetched at their pinned commits. Returns (dest, commit).
    """
    if repo.startswith("OWNER/"):
        raise ValueError('set REPO in the config cell to your GitHub repo ("owner/name")')
    if get_secret is None:
        from kaggle_secrets import UserSecretsClient

        get_secret = UserSecretsClient().get_secret
    token = get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {k: v for k, v in os.environ.items() if k not in _GIT_DEBUG_VARS}
    env.update(
        GIT_TERMINAL_PROMPT="0",
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}",
    )
    hide = (token, auth)
    del token, auth
    dest = pathlib.Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    _git(["init", "-q", str(dest)])
    _git(["remote", "add", "origin", f"https://github.com/{repo}.git"], cwd=dest)
    try:
        _git(["fetch", "-q", "--depth", "1", "origin", ref], cwd=dest, env=env, redact=hide)
    except RuntimeError as e:
        raise RuntimeError(
            f"{e}\n\nCould not fetch {repo} @ {ref}. Check: REPO/REF in the config cell; Internet on; "
            "GITHUB_TOKEN attached to this notebook and allowed to read this repo "
            "(fine-grained token: repo selected under 'Repository access', 'Contents: Read-only', not expired)."
        ) from None
    finally:
        del env, hide
    _git(["checkout", "-q", "FETCH_HEAD"], cwd=dest)
    _git(["submodule", "update", "--init", "--depth", "1", "-q"], cwd=dest)
    commit = _git(["rev-parse", "HEAD"], cwd=dest, capture=True)
    print(f"repo {repo} @ {ref} -> {commit}\n{_git(['submodule', 'status'], cwd=dest, capture=True)}")
    return dest, commit


def run_logged(cmd, log_path, cwd, extra_env=None):
    """Run `cmd`, streaming output to the notebook and appending it to `log_path`.

    Raises if the command fails, so a failed step stops "Save & Run All" visibly.
    """
    log_path = pathlib.Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", **(extra_env or {})}
    t0 = time.time()
    with open(log_path, "a") as log:
        log.write(f"\n### {time.strftime('%Y-%m-%d %H:%M:%S')} $ {' '.join(map(str, cmd))}\n")
        with subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
            for line in p.stdout:
                sys.stdout.write(line)
                log.write(line)
            rc = p.wait()
    print(f"\n[exit code {rc} after {(time.time() - t0) / 60:.1f} min; log: {log_path}]")
    if rc != 0:
        raise RuntimeError(f"step failed with exit code {rc}; see {log_path}")


CODE_DIR, COMMIT = clone_repo(REPO, REF)
PYTHONPATH = {"PYTHONPATH": f"{CODE_DIR}/src"}

In [ ]:
# pilot scripts import the Utonia, Sonata and Concerto inference packages (pinned submodules) and our src/
PILOT_PYTHONPATH = {"PYTHONPATH": ":".join(f"{CODE_DIR}/{p}" for p in ("src", "third_party/utonia", "third_party/sonata",
                                                                      "third_party/concerto"))}

In [ ]:
# choices from the pilot_02 run of 2026-10-26 11:47 UTC (decisions.md P6, results)
OCCLUSION = "v2"  # z-buffer variant (lifting.OCCLUSION): best F1 against the mesh in all three val scenes
CONCERTO = "concerto"  # concerto_large_origin: healthy in the half-room probe; "concerto_prerelease" is the alternative

In [ ]:
run_logged([sys.executable, "-u", "scripts/setup/install_env.py"],
           "/kaggle/working/logs/pilot_03_install.log", cwd=CODE_DIR, extra_env=PYTHONPATH)

In [ ]:
run_logged([sys.executable, "-u", "experiments/pilot_concat/step3_features.py", "--occlusion", OCCLUSION, "--concerto", CONCERTO],
           "/kaggle/working/logs/pilot_03_step3.log", cwd=CODE_DIR, extra_env=PILOT_PYTHONPATH)

In [ ]:
run_logged([sys.executable, "-u", "experiments/pilot_concat/step4_probes.py"],
           "/kaggle/working/logs/pilot_03_step4.log", cwd=CODE_DIR, extra_env=PILOT_PYTHONPATH)

In [ ]:
run_logged([sys.executable, "-u", "experiments/pilot_concat/step5_eval.py"],
           "/kaggle/working/logs/pilot_03_step5.log", cwd=CODE_DIR, extra_env=PILOT_PYTHONPATH)